# Paper Method Reproduction — Saha et al. 2025



## Section 0 — Setup


In [1]:
# Standard library
import os, sys, math, warnings
from pathlib import Path

# Data and ML libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.metrics import r2_score, mean_squared_error

# Reduce noise from warnings
warnings.filterwarnings("ignore")

# Set working directory to the bundle root (one level above notebooks/)
BUNDLE_ROOT = Path("..").resolve()
os.chdir(BUNDLE_ROOT)
print("Working directory:", Path.cwd())

# Runtime control: set True for fast demo, False for full paper protocol
DEMO_MODE = True #False
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


Working directory: /Users/robertwang/Documents/New project/baseline_composite_biomakers


## Section 2 — Feature Definitions and Combinations

In [2]:
# Feature domains (Melbourne v1 columns)
background = [
    'age_at_scan',
    'age_at_onset',
    'disease_duration',
    'disease_duration_to_10',
    'disease_duration_10_up',
    'sex',
    'GAA1',
    'GAA2',
]

structural = [
    'SCP_v1', 'MCP_v1', 'ICP_v1',
    'Midbrain_v1', 'Pons_v1', 'Medulla_v1',
    'AntCBLM_v1', 'SupPostCBLM_v1', 'InfPostCBLM_v1',
    'FlocCBLM_v1', 'VermisCBLM_v1',
]

diffusion = [
    'FASCP_v1', 'FAMCP_v1', 'FAICP_v1',
    'MDSCP_v1', 'MDMCP_v1', 'MDICP_v1',
    'ADSCP_v1', 'ADMCP_v1', 'ADICP_v1',
    'RDSCP_v1', 'RDMCP_v1', 'RDICP_v1',
]

qsm = []  # not available

# Combination definitions (paper Table 3); QSM combos skipped
combinations = [
    {"name": "background_only", "domains": [background], "skip": False},
    {"name": "structural_only", "domains": [structural], "skip": False},
    {"name": "diffusion_only", "domains": [diffusion], "skip": False},
    {"name": "qsm_only", "domains": [qsm], "skip": True},
    {"name": "all_neuroimaging", "domains": [structural, diffusion], "skip": False},
    {"name": "background_structural", "domains": [background, structural], "skip": False},
    {"name": "background_diffusion", "domains": [background, diffusion], "skip": False},
    {"name": "background_qsm", "domains": [background, qsm], "skip": True},
    {"name": "background_structural_diffusion", "domains": [background, structural, diffusion], "skip": False},
    {"name": "background_all_neuroimaging", "domains": [background, structural, diffusion, qsm], "skip": True},
]

print('Defined combinations:')
for c in combinations:
    print(c['name'], 'skip' if c['skip'] else 'run', 'features:', sum(len(d) for d in c['domains']))


Defined combinations:
background_only run features: 8
structural_only run features: 11
diffusion_only run features: 12
qsm_only skip features: 0
all_neuroimaging run features: 23
background_structural run features: 19
background_diffusion run features: 20
background_qsm skip features: 8
background_structural_diffusion run features: 31
background_all_neuroimaging skip features: 31


## Section 3 — QC Checks (Missingness / Outliers)

In [3]:
# Load Melbourne-only merged data for baseline paper run (v1 only)
melb_path = Path('data/processed/melbourne_frda_merged.csv')
assert melb_path.exists(), 'Run data_processing_merge.ipynb first to generate melbourne_frda_merged.csv'

merged_long = pd.read_csv(melb_path)
print('Loaded melbourne_frda_merged.csv:', merged_long.shape)

# Derive v1 background columns to match model expectations
merged_long['age_at_scan'] = merged_long.get('age1')
merged_long['disease_duration'] = merged_long.get('dur1')
merged_long['disease_duration_to_10'] = merged_long['disease_duration'].clip(upper=10)
merged_long['disease_duration_10_up'] = (merged_long['disease_duration'] - 10).clip(lower=0)


Loaded melbourne_frda_merged.csv: (26, 67)


In [4]:
# Missingness summary

def missingness_summary(df, cols):
    # Only keep columns that exist, warn if any are missing
    present = [c for c in cols if c in df.columns]
    missing = [c for c in cols if c not in df.columns]
    if missing:
        print("WARNING: missing columns:", missing)
    return df[present].isna().sum().sort_values(ascending=False)

print("Missingness (top 10):")
print(missingness_summary(merged_long, background + structural + diffusion).head(10))


Missingness (top 10):
age_at_scan       0
InfPostCBLM_v1    0
RDMCP_v1          0
RDSCP_v1          0
ADICP_v1          0
ADMCP_v1          0
ADSCP_v1          0
MDICP_v1          0
MDMCP_v1          0
MDSCP_v1          0
dtype: int64


In [5]:
# Tukey's Fences outlier filter (k=3) — features only

def tukey_outliers_mask(df, cols, k=3.0):
    mask = pd.Series(False, index=df.index)
    for col in cols:
        if col not in df.columns:
            continue
        series = df[col].dropna()
        if series.empty:
            continue
        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        lo = q1 - k * iqr
        hi = q3 + k * iqr
        mask |= (df[col] < lo) | (df[col] > hi)
    return mask


## Section 4 — Model Training (ElasticNet)
From paper protocol: 10×10 GroupKFold, grid over alpha (0..1) and lambda (0..20).


In [6]:
# Hyperparameter grid
if DEMO_MODE:
    alphas = np.linspace(0, 1, 3)        # tiny grid
    lambdas = np.linspace(0, 2, 5)
    n_splits = 3
    n_repeats = 2
else:
    alphas = np.linspace(0, 1, 11)       # 0..1 step 0.1
    lambdas = np.linspace(0, 20, 201)    # 0..20 step 0.1
    n_splits = 10
    n_repeats = 10

print("Grid size:", len(alphas) * len(lambdas))


Grid size: 15


In [7]:
# Cross-validation with per-fold hyperparameter search

def run_cv_for_combination(df, target_col, combo):
    feature_cols = []
    for domain in combo["domains"]:
        feature_cols.extend(domain)

    # Drop rows missing any features or target
    sub = df.dropna(subset=feature_cols + [target_col]).copy()
    n_before = len(sub)

    # Remove outliers (features only)
    outlier_mask = tukey_outliers_mask(sub, feature_cols, k=3.0)
    sub = sub[~outlier_mask].copy()
    n_after = len(sub)
    # Grouping column (subject ID)
    group_col = 'subject' if 'subject' in sub.columns else ('melb_id' if 'melb_id' in sub.columns else ('ID' if 'ID' in sub.columns else None))
    if group_col is None:
        raise KeyError('No subject/group id column found for GroupKFold')

    groups = sub[group_col].values
    X = sub[feature_cols].values
    y = sub[target_col].values

    all_preds = []
    all_true = []
    best_alphas = []
    best_l1s = []

    for r in range(n_repeats):
        rng = np.random.RandomState(RANDOM_SEED + r)
        unique_groups = np.unique(groups)
        rng.shuffle(unique_groups)

        # Shuffle rows by group order (paper repeats CV with reshuffled groups)
        group_order = {g: i for i, g in enumerate(unique_groups)}
        order = np.argsort([group_order[g] for g in groups])
        Xr, yr, gr = X[order], y[order], groups[order]

        gkf = GroupKFold(n_splits=n_splits)
        for train_idx, val_idx in gkf.split(Xr, yr, gr):
            X_train, X_val = Xr[train_idx], Xr[val_idx]
            y_train, y_val = yr[train_idx], yr[val_idx]

            # Standardize X and y within fold
            xs = StandardScaler()
            ys = StandardScaler()
            X_train_s = xs.fit_transform(X_train)
            X_val_s = xs.transform(X_val)
            y_train_s = ys.fit_transform(y_train.reshape(-1, 1)).ravel()

            best_rmse = np.inf
            best_pred = None
            best_a = None
            best_l = None

            # Grid search on this fold
            for a in alphas:
                for l in lambdas:
                    model = ElasticNet(alpha=l, l1_ratio=a, max_iter=5000, random_state=RANDOM_SEED)
                    model.fit(X_train_s, y_train_s)
                    y_pred_s = model.predict(X_val_s)
                    y_pred = ys.inverse_transform(y_pred_s.reshape(-1, 1)).ravel()

                    rmse = math.sqrt(mean_squared_error(y_val, y_pred))
                    if rmse < best_rmse:
                        best_rmse = rmse
                        best_pred = y_pred
                        best_a = a
                        best_l = l

            all_preds.extend(best_pred)
            all_true.extend(y_val)
            if best_a is not None and best_l is not None:
                best_l1s.append(best_a)
                best_alphas.append(best_l)

    r2 = r2_score(all_true, all_preds)
    rmse = math.sqrt(mean_squared_error(all_true, all_preds))
    # Summarize best params across folds (mode + mean)
    def _mode(vals):
        if not vals:
            return None
        return max(set(vals), key=vals.count)
    best_alpha_mode = _mode(best_alphas)
    best_l1_mode = _mode(best_l1s)
    best_alpha_mean = float(np.mean(best_alphas)) if best_alphas else None
    best_l1_mean = float(np.mean(best_l1s)) if best_l1s else None
    return r2, rmse, len(sub), n_before, n_after, best_alpha_mode, best_l1_mode, best_alpha_mean, best_l1_mean


In [8]:
# Run CV for all combinations and both targets
results = []

for target_col in ["FARS1", "SARA1"]:
    print("\n=== Target:", target_col, "===")
    for combo in combinations:
        if combo["skip"]:
            results.append({
                "target": target_col,
                "combination": combo["name"],
                "r2": np.nan,
                "rmse": np.nan,
                "n_rows": 0,
                "status": "skipped",
            })
            continue
        print("Running", target_col, combo["name"], "| features:", sum(len(d) for d in combo["domains"]))
        r2, rmse, n_rows, n_before, n_after, best_alpha_mode, best_l1_mode, best_alpha_mean, best_l1_mean = run_cv_for_combination(merged_long, target_col, combo)
        if n_before - n_after == 0:
            print("Outlier removal for %s / %s: removed 0 (none), left %d" % (combo["name"], target_col, n_after))
        else:
            print("Outlier removal for %s / %s: removed %d, left %d" % (combo["name"], target_col, n_before - n_after, n_after))
        results.append({
            "target": target_col,
            "combination": combo["name"],
            "r2": r2,
            "rmse": rmse,
            "lambda_mean": best_l1_mean,
            "alpha_mean": best_alpha_mean,
            #"lambda_mode": best_l1_mode,
            #"alpha_mode": best_alpha_mode,
            "n_rows": n_rows,
            "status": "completed",
        })

cv_df = pd.DataFrame(results)

# Show metrics + hyperparameters per combination
cols = [c for c in ["target","combination","r2","rmse","n_rows","lambda_mode","alpha_mode","lambda_mean","alpha_mean"] if c in cv_df.columns]
cv_df[cols]



=== Target: FARS1 ===
Running FARS1 background_only | features: 8
Outlier removal for background_only / FARS1: removed 3, left 23
Running FARS1 structural_only | features: 11
Outlier removal for structural_only / FARS1: removed 0 (none), left 26
Running FARS1 diffusion_only | features: 12
Outlier removal for diffusion_only / FARS1: removed 1, left 25
Running FARS1 all_neuroimaging | features: 23
Outlier removal for all_neuroimaging / FARS1: removed 1, left 25
Running FARS1 background_structural | features: 19
Outlier removal for background_structural / FARS1: removed 3, left 23
Running FARS1 background_diffusion | features: 20
Outlier removal for background_diffusion / FARS1: removed 4, left 22
Running FARS1 background_structural_diffusion | features: 31
Outlier removal for background_structural_diffusion / FARS1: removed 4, left 22

=== Target: SARA1 ===
Running SARA1 background_only | features: 8
Outlier removal for background_only / SARA1: removed 3, left 23
Running SARA1 structura

,target,combination,r2,rmse,n_rows,lambda_mean,alpha_mean
0,FARS1,background_only,0.323048,18.840900,23,0.000000,1.000000
1,FARS1,structural_only,0.230665,22.677239,26,0.166667,1.000000
2,FARS1,diffusion_only,-0.001517,25.104070,25,0.333333,0.333333
3,FARS1,qsm_only,NaN,NaN,0,NaN,NaN
4,FARS1,all_neuroimaging,0.199423,22.444852,25,0.500000,0.500000
5,FARS1,background_structural,0.303170,19.115524,23,0.000000,1.166667
6,FARS1,background_diffusion,0.377033,17.492061,22,0.000000,1.333333
7,FARS1,background_qsm,NaN,NaN,0,NaN,NaN
8,FARS1,background_structural_diffusion,0.384152,17.391828,22,0.000000,0.666667
9,FARS1,background_all_neuroimaging,NaN,NaN,0,NaN,NaN


In [9]:
# Save CV results
results_dir = Path("results")
results_dir.mkdir(parents=True, exist_ok=True)

cv_df[cv_df.target == "FARS"].to_csv(results_dir / "cv_performance_fars.csv", index=False)
cv_df[cv_df.target == "SARA"].to_csv(results_dir / "cv_performance_sara.csv", index=False)
print("Saved CV CSVs")


Saved CV CSVs


## Section 5 — Variable Importance (Bootstrap)
201 bootstraps at subject level; coefficients are standardised.


In [10]:
def bootstrap_importance(df, target_col, combo, n_boot=201):
    feature_cols = []
    for domain in combo["domains"]:
        feature_cols.extend(domain)

    sub = df.dropna(subset=feature_cols + [target_col]).copy()
    sub = sub[~tukey_outliers_mask(sub, feature_cols, k=3.0)].copy()

    group_col = "subject" if "subject" in sub.columns else ("melb_id" if "melb_id" in sub.columns else ("ID" if "ID" in sub.columns else None))
    if group_col is None:
        raise KeyError("No subject/group id column found for bootstrap")
    groups = sub[group_col].values
    unique_groups = np.unique(groups)

    X = sub[feature_cols].values
    y = sub[target_col].values

    coef_list = []

    for _ in range(n_boot):
        sampled_groups = np.random.choice(unique_groups, size=len(unique_groups), replace=True)
        mask = np.isin(groups, sampled_groups)
        Xb, yb = X[mask], y[mask]

        xs = StandardScaler()
        ys = StandardScaler()
        Xs = xs.fit_transform(Xb)
        ys_ = ys.fit_transform(yb.reshape(-1, 1)).ravel()

        # Ridge-like setting (l1_ratio=0) as in the paper's best model
        model = ElasticNet(alpha=1.3, l1_ratio=0.0, max_iter=5000, random_state=RANDOM_SEED)
        model.fit(Xs, ys_)
        coef_list.append(model.coef_)

    coef_arr = np.vstack(coef_list)
    summary = {
        "feature": feature_cols,
        "coef_mean": coef_arr.mean(axis=0),
        "coef_p2_5": np.percentile(coef_arr, 2.5, axis=0),
        "coef_p97_5": np.percentile(coef_arr, 97.5, axis=0),
        "pct_nonzero": (coef_arr != 0).mean(axis=0),
    }
    return pd.DataFrame(summary)


In [11]:
# Run bootstrap for primary combination only (background_structural_diffusion)
primary = [c for c in combinations if c["name"] == "background_structural_diffusion"][0]

importance_dir = Path("results/variable_importance")
importance_dir.mkdir(parents=True, exist_ok=True)

for target_col in ["FARS1", "SARA1"]:
    print("Bootstrap importance:", target_col)
    imp = bootstrap_importance(merged_long, target_col, primary, n_boot=201)
    imp.to_csv(importance_dir / f"importance_{target_col.lower()}_{primary['name']}.csv", index=False)


Bootstrap importance: FARS1
Bootstrap importance: SARA1


## Section 6 — Composite + Sensitivity
This follows the paper's composite construction and paired v1/v2 sensitivity analysis.


In [12]:
def compute_composite(df, target_col, combo, alpha=1.3, l1_ratio=0.0):
    feature_cols = []
    for domain in combo["domains"]:
        feature_cols.extend(domain)

    sub = df.dropna(subset=feature_cols + [target_col]).copy()
    sub = sub[~tukey_outliers_mask(sub, feature_cols, k=3.0)].copy()

    X = sub[feature_cols].values
    y = sub[target_col].values

    xs = StandardScaler()
    ys = StandardScaler()
    Xs = xs.fit_transform(X)
    ys_ = ys.fit_transform(y.reshape(-1, 1)).ravel()

    model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=5000, random_state=RANDOM_SEED)
    model.fit(Xs, ys_)

    # Convert standardised coefficients to raw scale
    coef_std = model.coef_
    feature_sd = xs.scale_
    coef_raw = coef_std / feature_sd

    composite = (np.abs(coef_raw) * X).sum(axis=1)
    sub = sub.copy()
    sub["composite"] = composite
    return sub


In [13]:
# Composite + sensitivity with subject-level LOO (OOF predictions)
primary = [c for c in combinations if c["name"] == "background_structural_diffusion"][0]

# Build v1/v2 feature lists
feature_cols_v1 = [f for domain in primary["domains"] for f in domain]
feature_cols_v2 = [f.replace("_v1", "_v2") if f.endswith("_v1") else f for f in feature_cols_v1]
base_cols = [f.replace("_v1", "") for f in feature_cols_v1]

# Identify subject id column
id_col = "subject" if "subject" in merged_long.columns else ("melb_id" if "melb_id" in merged_long.columns else ("ID" if "ID" in merged_long.columns else None))
if id_col is None:
    raise KeyError("No subject id column found for composite")

# Build long format (v1 + v2)

def build_long_format(df):
    df_v1 = df[[id_col] + feature_cols_v1 + ["FARS1"]].copy()
    df_v1.columns = [id_col] + base_cols + ["FARS"]
    df_v1["visit"] = 1

    df_v2 = df[[id_col] + feature_cols_v2 + ["FARS2"]].copy()
    df_v2.columns = [id_col] + base_cols + ["FARS"]
    df_v2["visit"] = 2

    return pd.concat([df_v1, df_v2], ignore_index=True)

long_df = build_long_format(merged_long)

# Subject-level LOO composite (OOF predictions)

def compute_composite_loocv(df_long, feature_cols, target_col, subject_col, alpha=1.3, l1_ratio=0.0):
    sub = df_long.dropna(subset=feature_cols + [target_col]).copy()
    sub = sub[~tukey_outliers_mask(sub, feature_cols, k=3.0)].copy()

    subjects = sub[subject_col].unique()
    results = []

    for subj in subjects:
        test_mask = sub[subject_col] == subj
        train_mask = ~test_mask

        X_train = sub.loc[train_mask, feature_cols].values
        y_train = sub.loc[train_mask, target_col].values
        X_test = sub.loc[test_mask, feature_cols].values

        xs = StandardScaler()
        ys = StandardScaler()
        X_train_s = xs.fit_transform(X_train)
        y_train_s = ys.fit_transform(y_train.reshape(-1, 1)).ravel()
        X_test_s = xs.transform(X_test)

        model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=5000, random_state=RANDOM_SEED)
        model.fit(X_train_s, y_train_s)

        y_pred_s = model.predict(X_test_s)
        y_pred = ys.inverse_transform(y_pred_s.reshape(-1, 1)).ravel()

        test_rows = sub.loc[test_mask].copy()
        test_rows["composite_pred"] = y_pred
        results.append(test_rows)

    return pd.concat(results, ignore_index=True)

result = compute_composite_loocv(long_df, base_cols, "FARS", id_col, alpha=1.3, l1_ratio=0.0)

# Pair v1/v2 predictions
v1_pred = result[result["visit"] == 1][[id_col, "composite_pred"]]
v2_pred = result[result["visit"] == 2][[id_col, "composite_pred"]]
paired = v1_pred.merge(v2_pred, on=id_col, suffixes=("_v1", "_v2")).dropna()

# Sensitivity metrics
diff = paired["composite_pred_v2"] - paired["composite_pred_v1"]
cohens_d = diff.mean() / diff.std(ddof=1)

from scipy.stats import ttest_rel
p_val = ttest_rel(paired["composite_pred_v2"], paired["composite_pred_v1"]).pvalue

print(f"Composite Cohen's d: {cohens_d:.3f}")
print(f"Paired t p-value:    {p_val:.4f}")

# Sensitivity summary table (Composite + FARS baseline)
mean_change = diff.mean()
sd_change = diff.std(ddof=1)

if "FARS1" in merged_long.columns and "FARS2" in merged_long.columns:
    fars_pair = merged_long[["FARS1", "FARS2"]].dropna()
    fars_diff = fars_pair["FARS2"] - fars_pair["FARS1"]
    fars_mean = fars_diff.mean()
    fars_sd = fars_diff.std(ddof=1)
    fars_d = fars_mean / fars_sd if fars_sd != 0 else float("nan")
else:
    fars_mean = fars_sd = fars_d = float("nan")

sensitivity_table = pd.DataFrame([
    {"biomarker": "Composite", "n": len(diff), "mean_change": mean_change, "sd_change": sd_change, "d_score": cohens_d},
    {"biomarker": "FARS", "n": len(fars_diff) if 'fars_diff' in locals() else 0, "mean_change": fars_mean, "sd_change": fars_sd, "d_score": fars_d},
])

print("Sensitivity table:")
print(sensitivity_table)


Composite Cohen's d: 0.393
Paired t p-value:    0.0726
Sensitivity table:
   biomarker   n  mean_change  sd_change   d_score
0  Composite  23     1.810265   4.604190  0.393178
1       FARS  26     6.819231   7.938615  0.858995


In [14]:
# Save sensitivity summary
sens = pd.DataFrame({
    "metric": ["cohens_d", "paired_t_p"],
    "value": [cohens_d, p_val],
})

sens.to_csv(Path("results") / "composite_sensitivity.csv", index=False)
print("Saved composite_sensitivity.csv")


Saved composite_sensitivity.csv
